In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB, ComplementNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score ,classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.cluster import FeatureAgglomeration
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from collections import Counter
import time
import warnings

## 1. Loading The Data
We start by loading the 20 newsgroups dataset's raw text data and insepct it to see what are we going to work with.

In [ ]:
# Load the data as is from the 20 newsgroups dataset.
train_data_raw = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)

Let's have a look at two randomly chosen example documents of the original raw data.

In [ ]:
# Save some example documents (chosen randomly, can change the index to see other documents).
example_document1 = train_data_raw.data[39]
example_document2 = train_data_raw.data[40]

print('1st Example:\n',example_document1)
print('2nd Example:\n',example_document2)

By examining the examples of the original raw data, We noticed that headers, footers, and quoted text contain a lot of irrelevant information that could mislead the classifier or information. Headers often include metadata such as sender names, email addresses, and server details, which do not contribute to the actual topic of discussion. Footers usually contain signatures, which may introduce bias, and quoted text repeats parts of previous messages, creating redundancy. Keeping these elements could cause the model to overfit to superficial patterns rather than meaningful content. 
To ensure the classifier learns from the content of the document itself, we will use the remove parameter to exclude these elements when loading the training and test data.

In [2]:
# Load the 20 Newsgroups dataset's training set with headers, footers, and quotes removed.
train_data = fetch_20newsgroups(subset='train', 
                                shuffle=True, 
                                random_state=42,
                                remove =("headers","footers","quotes"),
                               )

# Load the 20 Newsgroups dataset's testing set with headers, footers, and quotes removed.
test_data = fetch_20newsgroups(subset='test',
                               shuffle=True,
                               random_state=42,
                               remove=("headers","footers","quotes"),
                              )

# Get the raw documents and labels.
X_train, y_train = train_data.data, train_data.target
X_test, y_test = test_data.data, test_data.target

# Target names (class labels):
target_names = train_data.target_names
print('We can see that our 18,846 samples are divided as follows:')
print('Training data samples -',len(X_train))
print('Testing data samples -',len(X_test))
print('Number of classes -',len(target_names))


We can see that our 18,846 samples are divided as follows:
Training data samples - 11314
Testing data samples - 7532
Number of classes - 20


## 2. Data distribution
It's interesting to see how does the data is distributed to different classes, and how does the raw data looks like.

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count the number of files in each class for the training and test data
train_class_counts = Counter(y_train)
test_class_counts = Counter(y_test)

# Sort by descending count
sorted_counts = sorted(train_class_counts.items(), key=lambda x: x[1], reverse=True)
classes = [target_names[idx] for idx, _ in sorted_counts]
counts = [count for _, count in sorted_counts]

# Average files per class in train data
avg_files_per_class_train = np.mean(list(train_class_counts.values()))
print(f"Average number of files per class in training data: {avg_files_per_class_train:.2f}")

# Do the same for test data
sorted_counts_test = sorted(test_class_counts.items(), key=lambda x: x[1], reverse=True)
classes_test = [target_names[idx] for idx, _ in sorted_counts_test]
counts_test = [count for _, count in sorted_counts_test]

# Convert to DataFrames for easy plotting in Seaborn
df_train = pd.DataFrame({"Class": classes, "Count": counts})
df_test = pd.DataFrame({"Class": classes_test, "Count": counts_test})

# Plot training data
plt.figure(figsize=(8, 5))
sns.barplot(data=df_train, x="Count", y="Class", color="C2", dodge=False)
plt.xlabel("Number of Files", fontsize=12)
plt.ylabel("Class Name", fontsize=12)
plt.title("Training Data: Number of Files per Class", fontsize=14)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# Plot testing data
plt.figure(figsize=(8, 5))
sns.barplot(data=df_test, x="Count", y="Class", color="C3", dodge=False)
plt.xlabel("Number of Files", fontsize=12)
plt.ylabel("Class Name", fontsize=12)
plt.title("Testing Data: Number of Files per Class", fontsize=14)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

The data is quite balanced in the most part, However there are some classes("talk.religion.misc" (377 files), "talk.politics.misc" (465 files) and "alt.etheism" (480 files)) that have fewer files compared to the other classes which are around 600 files each. This might lead to higher accuracy on majority classes while underpreforming on minority classes when using Algorithms like Logistic Regression and SVM which optimize a global loss. same thing might happen with kNN and Decision Trees. 
We will see about that later in the project.

## 3. Preprocessing
We can see that our original raw data contains a lot of noise such as punctuation, numbers and special characters. 
As Tf-idf vectorizer will represent the data as a sparse matrix, if we wont remove the noise factors, they will add unnecessary dimensions that do not contribute to meaning and will increase computational cost without meaningful gains.
Also, extra noise can cause models to learn irrelevant patterns, making them less effective.
Another thing is that the data contains stop words(e.g., "the", "is", "and") which are common across all the data, meaning they do not help in distinguishing between categories so keeping them increase noise and will overshadow more meaningful words.

Concluding our wanted Preprocessing steps to execute:
1. Cleaning - Convert text to lowercase (treat "Apple" and "apple" as the same word) and remove punctuation, numbers and special characters.
2. Tokenization - Using tf-idf vectorized we will convert the raw data into vectors.
3. Stemming - converting words to their base form, Changing Changed Change will turn into Chang.

In [3]:
# Define a custom analyzer function that will be used to preprocess the text data.
def custom_analyzer(text):
    # 1. Cleaning & Lowercasing: Convert text to lowercase and remove non-word characters.
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    
    # 2. Tokenization: Tokenize the cleaned text.
    tokens = word_tokenize(text)
    # 3. Stopwords Removal: Filter out common English stopwords.
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    
    # 4. Stemming: Convert tokens to their base form using PorterStemmer.
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    
    return stemmed_tokens


# # Create a TF-IDF Vectorizer with the custom analyzer.
# tfidf_vect = TfidfVectorizer(
#     analyzer=custom_analyzer,  # use our custom analyzer function
#     max_df=0.5,                # ignore terms that appear in more than 50% of the documents
#     min_df=5,                   # ignore terms that appear in fewer than 5 documents
# )

# # Fit and transform the training data, then transform the testing data.
# X_train_tfidf = tfidf_vect.fit_transform(X_train) # fit - learn the vocabulary and idf, transform - converting documents into document-term matrix
# X_test_tfidf = tfidf_vect.transform(X_test) # transform - converting documents into document-term matrix based on the vocabulary and idf learned from the training data


count_vect = CountVectorizer(
    analyzer=custom_analyzer,  # use our custom analyzer function
    max_df=0.5,                # ignore terms that appear in more than 50% of the documents
    min_df=5,                   # ignore terms that appear in fewer than 5 documents
)

# Fit and transform the training data, then transform the testing data.
X_train_counts = count_vect.fit_transform(X_train)
X_test_counts = count_vect.transform(X_test)

print("CountVectorizer train data shape:", X_train_counts.shape)
print("CountVectorizer test data shape:", X_test_counts.shape)
# print("TF-IDF train data shape:", X_train_tfidf.shape)
# print("TF-IDF test data shape:", X_test_tfidf.shape)


CountVectorizer train data shape: (11314, 13171)
CountVectorizer test data shape: (7532, 13171)


## 4. Evaluation 
We want to see the performance of Logistic regression, Decision Tree, Random forest, LinearSVC, kNN and Naive Bayes on the data.
In order to do so we will wrap them all in a pipeline to run one by one in an organized way.

In [4]:
# Define the classifiers to test, the parameters each classifier gets are generic and not optimized.
classifiers = [
    # ('Logistic Regression', LogisticRegression(max_iter=10000)),
    ('Ridge Classifier', RidgeClassifier()),
    ('Decision Tree', DecisionTreeClassifier()),
    ('Random Forest', RandomForestClassifier()),
    # ('LinearSVC', LinearSVC()), 
    ('KNN', KNeighborsClassifier(metric="cosine")), # by default the metric is minkowski
    ('Complement Naive Bayes', ComplementNB())
]

classifiers_predictions = {}

# Function to evaluate a classifier and return metrics
def get_classifier_metrics(clf, X_train, X_test, y_train, y_test, clf_name, to_fit=True):
    start_time = time.time()
    
    # Train the classifier if asked to.
    if to_fit == True:
        clf.fit(X_train, y_train)
    
    training_time = time.time() - start_time
    # Make predictions.
    if clf_name == 'Logistic Regression':
        y_pred = clf.predict(X_test.toarray())
    else:
        y_pred = clf.predict(X_test)
    classifiers_predictions[clf_name] = y_pred
    accuracy = accuracy_score(y_test, y_pred)*100
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)*100
    recall = recall_score(y_test, y_pred, average='weighted')*100
    f1 = f1_score(y_test, y_pred, average='weighted')*100
    
    return {
        'Classifier': clf_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Training Time (s)': training_time,
        'y_pred': y_pred
    }

def evaluate_classifiers(classifiers, X_train, X_test, y_train, y_test, dataset_label):
    """
    Evaluates each classifier on the given dataset, records the results,
    Returns the results.
    """
    results = []

    print(f"\n=== Running classifiers on {dataset_label} data ({X_train.shape[1]} features) ===\n")
    for clf_name, clf in classifiers:
        try:
            result = get_classifier_metrics(clf, X_train, X_test, y_train, y_test, clf_name)
            result['Dimensions'] = 'Original (No Reduction)'
            result['n_components'] = X_train.shape[1]
            results.append(result)
            print(f"Completed: {clf_name} on {dataset_label} data")
        except Exception as e:
            print(f"Error with {clf_name} on {dataset_label} data: {e}")
    return results

def format_results_table(df, groupby='n_components'):
    # Sort the results
    result_df = df.sort_values(by=[groupby, 'Accuracy'], ascending=[True, False]).copy()
    if 'y_pred' in result_df.columns:
        result_df.drop(columns=['y_pred'],inplace=True)
    return result_df


now let's evaluate the classifiers with tf-idf and countVectorizer, to see which one gives the better results.

In [ ]:
# Evaluate classifiers on both TF-IDF and Count datasets
# results_tfidf = evaluate_classifiers(classifiers, X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")
results_count = evaluate_classifiers(classifiers, X_train_counts, X_test_counts, y_train, y_test, "Count")

# Convert the results list to DataFrames
# results_df_tfidf = pd.DataFrame(results_tfidf)
results_df_count = pd.DataFrame(results_count)

# Filter results for original dimensionality
# original_results_tfidf = results_df_tfidf[results_df_tfidf['Dimensions'] == 'Original (No Reduction)']
original_results_count = results_df_count[results_df_count['Dimensions'] == 'Original (No Reduction)']

# Format and display the TF-IDF results table
# styled_df_tfidf = (
#     format_results_table(original_results_tfidf, groupby='n_components').style \
#     .set_caption("Results for Original TF-IDF Data") \
#     .format({
#         'Accuracy': '{:.2f}%',
#         'Precision': '{:.2f}%',
#         'Recall': '{:.2f}%',
#         'F1-Score': '{:.2f}%'
#     }))
# display(styled_df_tfidf.hide())

# Format and display the Count Vectorizer results table
styled_df_count = format_results_table(original_results_count, groupby='n_components').style \
    .set_caption("Results for Original Count Vectorizer Data") \
    .format({
        'Accuracy': '{:.2f}%',
        'Precision': '{:.2f}%',
        'Recall': '{:.2f}%',
        'F1-Score': '{:.2f}%'
    })

display(styled_df_count.hide())



=== Running classifiers on Count data (13171 features) ===



{TODO}

## Over sampling

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

print("Original distribution:", Counter(y_train))
# Let's identify the class indices for alt.atheism, talk.politics.misc, talk.religion.misc
# The 20 Newsgroups target names are in 'train_data.target_names'.
# For example:
minority_class_names = ["alt.atheism", "talk.politics.misc", "talk.religion.misc"]
minority_class_indices = [
    train_data.target_names.index(name) for name in minority_class_names
]

# We want each of these classes to have 565 samples
TARGET_COUNT = 565

# Create a sampling_strategy dict where each class keeps its original count,
# except for the 3 minority classes we want to boost to 565.
counts = Counter(y_train)
sampling_strategy = dict(counts)  # start with the original distribution
for cls_idx in minority_class_indices:
    sampling_strategy[cls_idx] = TARGET_COUNT

# Initialize the RandomOverSampler with our custom strategy
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)

# Oversample the training data
X_train_tfidf_over, y_train_over = ros.fit_resample(X_train_tfidf, y_train)

print("New distribution:", Counter(y_train_over))

In [ ]:
# Evaluate classifiers on both TF-IDF and Count datasets
results_tfidf_oversmpld = evaluate_classifiers(classifiers, X_train_tfidf_over, X_test_tfidf, y_train_over, y_test, "Over Sampled")

# Convert the results list to DataFrames
results_df_tfidf_oversmpld = pd.DataFrame(results_tfidf_oversmpld)

# Filter results for original dimensionality
original_results_tfidf_oversmpld = results_df_tfidf_oversmpld[results_df_tfidf_oversmpld['Dimensions'] == 'Original (No Reduction)']

# Format and display the TF-IDF results table
styled_df_tfidf_oversmpld = (
    format_results_table(original_results_tfidf_oversmpld, groupby='n_components').style \
    .set_caption("Results for Over-Sampled Data") \
    .format({
        'Accuracy': '{:.2f}%',
        'Precision': '{:.2f}%',
        'Recall': '{:.2f}%',
        'F1-Score': '{:.2f}%'
    }))
display(styled_df_tfidf_oversmpld.hide())

## Feature Agglomeration
We want to check if reducing the feature space from 13171 to smaller space using feature agglomeration might help us preform better.
In order to do so we need to turn our sparse data to dense data and rerun our classifiers.

In [ ]:
# --- Feature Agglomeration ---
# Reduce the number of features by clustering similar features together.
# Note: FeatureAgglomeration requires dense input, so we convert the TF-IDF matrices.
# Adjust n_clusters to suit your balance of dimensionality vs. information.
n_clusters = 1704  # You can experiment with this value
fa = FeatureAgglomeration(n_clusters=n_clusters)

# Convert sparse matrices to dense arrays before agglomeration.
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

X_train_reduced = fa.fit_transform(X_train_dense)
X_test_reduced = fa.transform(X_test_dense)

In [ ]:
# Evaluate classifiers on both TF-IDF and Count datasets
results_tfidf_reduced = evaluate_classifiers(classifiers, X_train_reduced, X_test_reduced, y_train, y_test, "TF-IDF")

# Convert the results list to DataFrames
results_df_tfidf_reduced = pd.DataFrame(results_tfidf_reduced)

# Filter results for original dimensionality
original_results_tfidf_reduced = results_df_tfidf_reduced[results_df_tfidf_reduced['Dimensions'] == 'Original (No Reduction)']

# Format and display the TF-IDF results table
styled_df_tfidf_reduced = (
    format_results_table(original_results_tfidf_reduced, groupby='n_components').style \
    .set_caption("Results for Original TF-IDF Data") \
    .format({
        'Accuracy': '{:.2f}%',
        'Precision': '{:.2f}%',
        'Recall': '{:.2f}%',
        'F1-Score': '{:.2f}%'
    }))
display(styled_df_tfidf_reduced.hide())

In [ ]:

for name, _ in classifiers:
    fig, ax = plt.subplots(figsize=(12, 12))  # Larger figure for better readability
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        classifiers_predictions[name],
        display_labels=target_names,  # Use your list of class names
        cmap='Blues',                 # Friendlier color map
        values_format='d',            # Show integer counts
        xticks_rotation='vertical',   # Rotate x-ticks if labels overlap
        ax=ax,
        colorbar=False
    )
    ax.set_title(f"Confusion Matrix: {name} (TF-IDF)", fontsize=16)
    ax.set_xlabel("Predicted Label", fontsize=14)
    ax.set_ylabel("True Label", fontsize=14)
    plt.tight_layout()  # Ensures labels and titles fit nicely
    plt.show()

## Dimensionality Reduction
As we know, our training data contains 13171 features(possible terms) in the sparse matrix. 
Most documents will contain only a small fraction of the possible terms, with most entries being zero.
We want to check if dimension reduction might help us getting rid of less important features that might add noise, prevent overfitting, prevent models from memorizing data patterns that don't generalize, etc. It seems quite logical to us that maintaining a reasonable amount of explained variance (70%) of the original data after the dimensionality reduction will help us get better results, so lets check what would be the number of features that which will give us that 70% explained variance.

In [ ]:
#reducing dimensions

import numpy as np

class ShiftTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Determine the shift value needed to make all values non-negative
        self.shift_ = -X.min() if X.min() < 0 else 0
        return self

    def transform(self, X, y=None):
        return X + self.shift_
    
# --- Optimal n_components Selection via Explained Variance Analysis ---
# Set a high number of components to analyze the explained variance
max_components = 2000
svd_full = TruncatedSVD(n_components=max_components, random_state=42)
svd_full.fit(X_train_tfidf)
cumulative_variance = np.cumsum(svd_full.explained_variance_ratio_)
# Set threshold for desired explained variance (e.g., 70%)
threshold = 0.70
optimal_n_components = np.argmax(cumulative_variance >= threshold) + 1  # +1 because index starts at 0

plt.close('all')
print(f"\nOptimal n_components based on {threshold*100:.0f}% explained variance: {optimal_n_components}")
# Optional: Plot the cumulative explained variance curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, max_components + 1), cumulative_variance, marker='o', markersize=2)
plt.axhline(y=threshold, color='r', linestyle='--')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by TruncatedSVD Components')
plt.grid(True)
plt.show()

We got that 1704 components will get us that 70%, so lets run TruncatedSVD with `n_components=1704` and put our classifiers to the test.

In [ ]:
# --- SVD Pipeline Testing with Optimal n_components and other values ---
svd_results = []

print(f"\n=== Running SVD with n_components={optimal_n_components} ===\n")

# Create SVD pipeline
svd_pipeline = make_pipeline(
    TruncatedSVD(n_components=optimal_n_components, random_state=42),
    ShiftTransformer(),          # Shift data to remove negatives
    Normalizer(copy=False)       # Normalizer is important for kNN and Naive Bayes
)

# Apply SVD transformation
X_train_svd = svd_pipeline.fit_transform(X_train_tfidf)
X_test_svd = svd_pipeline.transform(X_test_tfidf)


svd_results = evaluate_classifiers(
    classifiers, X_train_svd, X_test_svd, y_train, y_test, "SVD Reduced"
)
results_df_svd = pd.DataFrame(svd_results)
results_df_svd['Dimensions'] = "SVD Reduced"
styled_df_svd = (
    format_results_table(results_df_svd, groupby='n_components').style
    .set_caption("Results for SVD Reduced Data")
    .format({
        'Accuracy': '{:.2f}%',
        'Precision': '{:.2f}%',
        'Recall': '{:.2f}%',
        'F1-Score': '{:.2f}%'
    })
)
display(styled_df_svd)

Although we expected that reducing the dimensionality from 13,171 to 1,704 features might improve performance and lower training times, we found the opposite. First, many of the chosen classifiers (e.g., Multinomial Naive Bayes, Complement Naive Bayes, and LinearSVC) handle high-dimensional, sparse data very effectively for text tasks. When we apply TruncatedSVD, we risk losing valuable discriminative information, causing a drop in accuracy.

Second, after SVD, the data becomes dense rather than sparse, which can actually slow down training for certain models, such as tree-based classifiers (Decision Tree, Random Forest) and KNN. This overhead can outweigh the benefit of having fewer dimensions. So while dimensionality reduction can be beneficial in some contexts, it can also degrade both speed and accuracy when working with text data and classifiers that naturally handle high-dimensional, sparse input.



## Hyperparameter Optimization Using GridSearchCV
The parameters we initially gave to the classifiers where generic and not optimized, so in this stage we will use GridSearchCV in order to find the optimal parameters for each classifier so we can get the best out of them and we will use them from now on.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define classifiers along with their parameter grids
classifiers_to_opt = [
    ('Logistic Regression', 
     LogisticRegression(random_state=42, max_iter=1000),
     {'C': [0.1, 1, 10, 100],
      'solver': ['lbfgs', 'liblinear']}),
    
    ('Decision Tree', 
     DecisionTreeClassifier(random_state=42),
     {'max_depth': [None, 5, 10, 20],
      'min_samples_split': [2, 5, 10],
      'min_samples_leaf': [1, 2, 4],
      'max_features': [None, 'sqrt', 'log2']}),
    
    ('Random Forest', 
     RandomForestClassifier(random_state=42),
     {'n_estimators': [20, 50, 100],
      'max_features': ['sqrt', 'log2'],
      'max_depth': [None, 5, 10, 20]}),
    
    ('LinearSVC', 
     LinearSVC(random_state=42, dual=True, max_iter=10000),
     {'C': [0.01, 0.1, 1, 10, 100]}),
    
    ('KNN', 
     KNeighborsClassifier(),
     {'n_neighbors': [7, 9, 11, 13],
      'weights': ['uniform', 'distance'],
      'metric': ['euclidean', 'manhattan', 'cosine']}),
    
    ('Naive Bayes', 
    #  MultinomialNB(),
     ComplementNB(),
    {'alpha': [0.1, 1, 10]}),

    ('Ridge Classifier',
     RidgeClassifier(random_state=42, tol=1e-2,solver="sparse_cg"),
     {'alpha': [0.1, 1, 10]})
]

# Dictionary to store the grid search results for each classifier
grid_search_results = {}

# Loop over classifiers and perform grid search
for name, clf, param_grid in classifiers_to_opt:
    print(f"Running GridSearchCV for {name}...")
    grid_search = GridSearchCV(clf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train_tfidf, y_train)
    print(f"Best parameters for {name}: {grid_search.best_params_}")
    print(f"Best cross-validation score for {name}: {grid_search.best_score_:.4f}\n")
    grid_search_results[name] = grid_search

In [ ]:
# Evaluate each optimized (best) classifier on the test set using your evaluation functions
optimized_results = []
optimized_confusion_figures = []
print("\n=== Evaluating Optimized Classifiers on TF-IDF Data ===\n")
for clf_name, grid in grid_search_results.items():
    best_model = grid.best_estimator_ # This model has the optimal parameters
    result = get_classifier_metrics(best_model, X_train_tfidf, X_test_tfidf, y_train, y_test, clf_name, to_fit=False)
    result['Best Parameters'] = grid.best_params_
    result['Dimensions'] = 'Original (No Reduction)'
    result['n_components'] = X_train_tfidf.shape[1]
    optimized_results.append(result)
    print(f"Completed: {clf_name} (Optimized)")
    # fig = plot_confusion_matrix(y_test, result['y_pred'], clf_name, "TF-IDF (Optimized)")
    # optimized_confusion_figures.append(fig)

# Create and display a formatted results table for the optimized classifiers
results_df_optimized = pd.DataFrame(optimized_results)
styled_df_optimized = format_results_table(results_df_optimized, groupby='n_components').style \
    .set_caption("Results for Optimized TF-IDF Data") \
    .format({
        'Accuracy': '{:.2f}%',
        'Precision': '{:.2f}%',
        'Recall': '{:.2f}%',
        'F1-Score': '{:.2f}%'
    })

styled_df_optimized.hide()
display(styled_df_optimized.hide())

## Confusion
It is interesting to see which classes are causing the classifiers to make mistakes and also we want to see how the classifiers did with the minority classes compared to the majority ones.

In [ ]:
# for fig in optimized_confusion_figures:
#     display(fig)


Lets use this number of components that achieves 70% explained variance and see how our results will be affected.